# 5-Class Cancer Classification Pipeline Debug Notebook

Run this notebook cell by cell so you can inspect paths, class counts, fold splits, training logs, and evaluation outputs.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

import torch
import pandas as pd

# Make the local package importable when the notebook is run from the notebook folder.
PROJECT_ROOT = Path(r"G:\\04PhD\\03Study\\05SelfLearn\\shilab-cancer-classifier")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shilab_cancer_classifier import (
    CANCER_CLASSES,
    NUM_CLASSES,
    SUPPORTED_MODELS,
    evaluate_all_models,
    prepare_cross_validation,
    split_dataset,
    train_all_models,
)

print("Project root:", PROJECT_ROOT)
print("Classes:", CANCER_CLASSES)
print("NUM_CLASSES:", NUM_CLASSES)
print("CUDA available:", torch.cuda.is_available())

## 1. Configure Experiment

For quick debugging, start with one model and a small epoch count. Increase these after the pipeline runs correctly.

In [ ]:
RAW_DATA_DIR = Path(r"H:\\0UR_classification_model\\raw_data")
BASE_OUTPUT_DIR = Path(r"H:\\0UR_classification_model")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_DIR = BASE_OUTPUT_DIR / f"experiment_debug_{timestamp}"

SPLIT_DATA_DIR = EXPERIMENT_DIR / "split_data"
CV_DATA_DIR = EXPERIMENT_DIR / "cross_validation_data"
MODELS_DIR = EXPERIMENT_DIR / "train_val_models"
RESULTS_DIR = EXPERIMENT_DIR / "evaluation_results"
TRAIN_VAL_DIR = SPLIT_DATA_DIR / "train_val"
TEST_DIR = SPLIT_DATA_DIR / "test"

NUM_FOLDS = 5
NUM_EPOCHS = 50  
BATCH_SIZE = 16
INITIAL_LR = 0.001
PATIENCE = 10
NUM_WORKERS = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODELS_TO_TRAIN = ["ResNet50"]
# MODELS_TO_TRAIN = SUPPORTED_MODELS

RUN_SPLIT = True
RUN_CV = True
RUN_TRAIN = True
RUN_EVAL = True

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print("Experiment dir:", EXPERIMENT_DIR)
print("Device:", DEVICE)
print("Models:", MODELS_TO_TRAIN)

## 2. Check Raw Data Folders

In [ ]:
if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(f"Raw data directory does not exist: {RAW_DATA_DIR}")

missing_classes = [
    class_name for class_name in CANCER_CLASSES
    if not (RAW_DATA_DIR / class_name).exists()
]
if missing_classes:
    raise FileNotFoundError(f"Missing class folders: {missing_classes}")

valid_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
raw_counts = {}
for class_name in CANCER_CLASSES:
    class_dir = RAW_DATA_DIR / class_name
    raw_counts[class_name] = sum(
        1 for p in class_dir.iterdir()
        if p.is_file() and p.suffix.lower() in valid_exts
    )

raw_counts_df = pd.DataFrame.from_dict(raw_counts, orient="index", columns=["count"])
display(raw_counts_df)

if raw_counts_df["count"].min() < NUM_FOLDS:
    raise ValueError(
        f"Smallest class has {raw_counts_df['count'].min()} images, "
        f"which is less than NUM_FOLDS={NUM_FOLDS}."
    )

## 3. Split Train/Test

In [ ]:
if RUN_SPLIT:
    split_result = split_dataset(
        raw_data_dir=str(RAW_DATA_DIR),
        class_names=CANCER_CLASSES,
        output_dir=str(SPLIT_DATA_DIR),
        split_ratio=0.9,
        random_seed=42,
    )
    print("Split log:", split_result["log_path"])
else:
    print("Skipping split step.")

In [ ]:
split_counts = []
for split_name, split_dir in [("train_val", TRAIN_VAL_DIR), ("test", TEST_DIR)]:
    for class_name in CANCER_CLASSES:
        class_dir = split_dir / class_name
        count = 0
        if class_dir.exists():
            count = sum(1 for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_exts)
        split_counts.append({"split": split_name, "class": class_name, "count": count})

split_counts_df = pd.DataFrame(split_counts)
display(split_counts_df.pivot(index="class", columns="split", values="count"))

## 4. Prepare Stratified Cross Validation

In [ ]:
if RUN_CV:
    cv_result = prepare_cross_validation(
        data_path=str(TRAIN_VAL_DIR),
        output_base=str(CV_DATA_DIR),
        n_splits=NUM_FOLDS,
        random_state=42,
    )
    print("CV log:", cv_result["log_path"])
    display(cv_result["df_summary"])
else:
    print("Skipping cross-validation step.")

## 5. Train Models

In [ ]:
if RUN_TRAIN:
    train_result = train_all_models(
        model_list=MODELS_TO_TRAIN,
        cv_data_dir=str(CV_DATA_DIR),
        output_dir=str(MODELS_DIR),
        num_folds=NUM_FOLDS,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        initial_lr=INITIAL_LR,
        device=DEVICE,
        num_classes=NUM_CLASSES,
        num_workers=NUM_WORKERS,
        patience=PATIENCE,
    )
    print("Training output:", train_result["output_dir"])
else:
    print("Skipping training step.")

In [ ]:
model_files = []
if MODELS_DIR.exists():
    model_files = sorted(MODELS_DIR.rglob("*.pth"))
print(f"Found {len(model_files)} model weight files")
for path in model_files[:10]:
    print(path)

## 6. Evaluate Models

In [ ]:
if RUN_EVAL:
    eval_result = evaluate_all_models(
        model_list=MODELS_TO_TRAIN,
        cv_data_dir=str(CV_DATA_DIR),
        test_dir=str(TEST_DIR),
        models_dir=str(MODELS_DIR),
        output_dir=str(RESULTS_DIR),
        num_folds=NUM_FOLDS,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        num_classes=NUM_CLASSES,
        device=DEVICE,
        save_summary=True,
    )
    print(eval_result)
else:
    print("Skipping evaluation step.")

## 7. Inspect Evaluation Outputs

In [ ]:
summary_path = RESULTS_DIR / "all_models_summary.csv"
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    display(summary_df)
else:
    print("Summary file not found:", summary_path)

In [ ]:
probability_files = sorted(RESULTS_DIR.rglob("*_probabilities.csv")) if RESULTS_DIR.exists() else []
print(f"Found {len(probability_files)} probability files")
if probability_files:
    sample_prob_df = pd.read_csv(probability_files[0])
    print(probability_files[0])
    display(sample_prob_df.head())